In [7]:
import spacy
# Cargar modelo ahora en español 
try:
    nlp = spacy.load("es_core_news_sm")
except OSError:
    print("Descargando modelo...")
    from spacy.cli import download
    download("es_core_news_sm")
    nlp = spacy.load("es_core_news_sm")

# TEXTO DE ENTRADA 
with open("cap1_harry_potter.txt", "r", encoding="utf-8") as f:
    texto_principito = f.read()

# Ahora 'texto_principito' contiene todo el texto del archivo
print(f"Texto cargado con éxito. Longitud: {len(texto_principito)} caracteres.")

# 1. TOKENIZACIÓN
# SpaCy procesa el texto y crea el objeto 'doc' lleno de metadatos
doc = nlp(texto_principito)

# Mostrar los primeros 15 tokens para entender cómo "ve" la máquina el texto
print(f"--- 1. Tokenización (Total tokens: {len(doc)}) ---")
print([token.text for token in doc][:20]) 

Texto cargado con éxito. Longitud: 27573 caracteres.
--- 1. Tokenización (Total tokens: 5666) ---
['El', 'señor', 'y', 'la', 'señora', 'Dursley', ',', 'que', 'vivían', 'en', 'el', 'número', '4', 'de', 'Privet', 'Drive', ',', 'estaban', 'orgullosos', 'de']


In [8]:
# 2. FILTRADO DE STOP WORDS
# Separamos lo que aporta valor semántico del "pegamento" gramatical

tokens_relevantes = []
tokens_ruido = []

for token in doc:
    # Filtramos si es stopword o si es puntuación
    if not token.is_stop and not token.is_punct and token.text.strip():
        tokens_relevantes.append(token.text)
    elif token.is_stop:
        tokens_ruido.append(token.text)

print(f"\n--- 2. Filtrado de Stop Words ---")
print(f"Palabras eliminadas (Ruido): {tokens_ruido[:10]}...")
print(f"Palabras conservadas (Contenido): {tokens_relevantes[:10]}...")
print(f"Reducción de tamaño: de {len(doc)} a {len(tokens_relevantes)} tokens.")


--- 2. Filtrado de Stop Words ---
Palabras eliminadas (Ruido): ['El', 'y', 'la', 'que', 'en', 'el', 'de', 'estaban', 'de', 'decir']...
Palabras conservadas (Contenido): ['señor', 'señora', 'Dursley', 'vivían', 'número', '4', 'Privet', 'Drive', 'orgullosos', 'normales']...
Reducción de tamaño: de 5666 a 1936 tokens.


In [9]:
# 3. LEMATIZACIÓN Y NORMALIZACIÓN FINAL
# Reducimos las palabras a su raíz (Lema) y estandarizamos a minúsculas
# Objetivo: Que "hablo", "hablaré" y "habla" cuenten como el mismo concepto: "hablar"

tokens_normalizados = []
cambios_interesantes = []

for token in doc:
    # Aplicamos los mismos filtros de calidad que en el paso 2
    if not token.is_stop and not token.is_punct and token.text.strip():
        
        # AQUÍ OCURRE LA MAGIA: 
        # 1. Extraemos el lema (token.lemma_)
        # 2. Convertimos a minúsculas (.lower())
        lema = token.lemma_.lower()
        tokens_normalizados.append(lema)
        
        # Para fines educativos: Guardamos casos donde la palabra cambió drásticamente
        # Ej: "fui" -> "ir"
        if token.text.lower() != lema:
            cambios_interesantes.append(f"{token.text} ➡ {lema}")

print(f"\n--- 3. Lematización y Normalización ---")
print(f"Total de tokens procesados: {len(tokens_normalizados)}")
print(f"Ejemplos de transformaciones (Palabra original ➡ Lema):")
# Mostramos solo los primeros 5 cambios para no saturar la pantalla
print(cambios_interesantes[:10]) 

print(f"\nResultado final (Primeros 10 tokens):")
print(tokens_normalizados[:10])


--- 3. Lematización y Normalización ---
Total de tokens procesados: 1936
Ejemplos de transformaciones (Palabra original ➡ Lema):
['vivían ➡ vivir', 'orgullosos ➡ orgulloso', 'normales ➡ normal', 'personas ➡ persona', 'esperaría ➡ esperar', 'relacionadas ➡ relacionado', 'tales ➡ tal', 'tonterías ➡ tontería', 'llamada ➡ llamado', 'fabricaba ➡ fabricar']

Resultado final (Primeros 10 tokens):
['señor', 'señora', 'dursley', 'vivir', 'número', '4', 'privet', 'drive', 'orgulloso', 'normal']


In [10]:
import pandas as pd
from nltk.stem import SnowballStemmer

# Configuración del Stemmer (NLTK)
stemmer = SnowballStemmer("spanish")
# Comparativa lado a lado
data_comparativa = []

for token in doc:
    # Solo analizamos palabras, ignoramos puntuación y espacios para claridad
    if not token.is_punct and not token.is_space:
        
        # A. STEMMING (Corte de sufijos)
        raiz_stem = stemmer.stem(token.text)
        
        # B. LEMATIZACIÓN (Análisis morfológico de SpaCy)
        # Nota: SpaCy usa el contexto para saber si 'fui' es 'ir' o 'ser'
        lema = token.lemma_
        
        data_comparativa.append({
            "Original": token.text,
            "Stemming (Corte)": raiz_stem,
            "Lematización (Diccionario)": lema,
            "¿Coinciden?": raiz_stem == lema
        })

# Crear DataFrame para visualizar
df = pd.DataFrame(data_comparativa)

print(f"\n--- 3. Stemming vs Lematización ---")
# Mostramos palabras interesantes donde se vea la diferencia
# Buscamos verbos conjugados o plurales
palabras_interesantes = ["hombres", "olvidado", "eres", "domesticado", "invisible", "ojos"]
filtro = df[df["Original"].isin(palabras_interesantes)]

print(filtro.to_string(index=False))

print("\n--- Visualización completa de los primeros 10 tokens ---")
print(df.head(10).to_string(index=False))


--- 3. Stemming vs Lematización ---
Original Stemming (Corte) Lematización (Diccionario)  ¿Coinciden?
olvidado            olvid                    olvidar        False
    ojos             ojos                        ojo        False
    ojos             ojos                        ojo        False
    ojos             ojos                        ojo        False
    ojos             ojos                        ojo        False
    ojos             ojos                        ojo        False
    ojos             ojos                        ojo        False
    ojos             ojos                        ojo        False
    ojos             ojos                        ojo        False
    ojos             ojos                        ojo        False

--- Visualización completa de los primeros 10 tokens ---
Original Stemming (Corte) Lematización (Diccionario)  ¿Coinciden?
      El               el                         el         True
   señor            señor                      